In [1]:
pip install pyomo


   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.9 MB 2.1 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/3.9 MB 2.2 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/3.9 MB 2.3 MB/s eta 0:00:02
   --------------------- ------------------ 2.1/3.9 MB 2.3 MB/s eta 0:00:01
   -------------------------- ------------- 2.6/3.9 MB 2.3 MB/s eta 0:00:01
   ------------------------------- -------- 3.1/3.9 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------  3.9/3.9 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 2.5 MB/s  0:00:01

   ---------------------------------------- 0/2 [ply]
   -------------------- ------------------- 1/2 [pyomo]
   -------------------- ------------------- 1/2 [pyomo]
   -------------------- ------------------- 1/2 [pyomo]
   -------------------- ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
pip install pandas openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [3]:
import pyomo.environ as pyo

PARÁMETROS DEL PROBLEMA

In [61]:
I = 143  #NODOS FACTIBLS POR COMUNA
J = 143
K = 2
I = range(1, I+1)   # Conjunto de i
J = range(1, J+1)   # Conjunto de j
K = range(1, K+1)   # Conjunto de k


import pandas as pd
df2 = pd.read_csv(r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\DISTANCIAS\17mutis.csv", sep=",")
df2


Ck = {k: val for k, val in zip(K, [376.5, 1061.2])}
Ct = 1
# Access the correctly parsed columns for Dij
Dij = {(int(row.i), int(row.j)): float(row.Dij) for _, row in df2.iterrows()}
Mk = {k: val for k, val in zip(K, [1, 3])}
N = 55

In [5]:
df2


,i,j,Dij,lat_i,lon_i,lat_j,lon_j
0,2,19,71.31,7.119031,-73.125987,7.119631,-73.126225
1,3,4,75.38,7.120337,-73.128172,7.120589,-73.127538
2,3,10,61.54,7.120337,-73.128172,7.120855,-73.128375
3,4,16,64.56,7.120589,-73.127538,7.121141,-73.127728
4,5,18,70.32,7.117911,-73.130927,7.117689,-73.131524
...,...,...,...,...,...,...,...
62,80,81,88.82,7.120592,-73.123629,7.120872,-73.122875
63,81,82,87.68,7.120872,-73.122875,7.121143,-73.122129
64,84,85,83.10,7.115045,-73.123897,7.115713,-73.124241
65,86,87,60.58,7.114891,-73.125676,7.114703,-73.125161


In [62]:
model = pyo.ConcreteModel()

model.X = pyo.Var(I, J, within=pyo.Binary)
model.Y = pyo.Var(I, K, within=pyo.Binary)

In [78]:
def obj_rule(m):
    #return sum(Ck[k]*m.Y[i,k] for i in I for k in K) - Ct * ( sum(m.X[i,j]*Dij[(i,j)] for (i,j) in Dij))
    #return Ct * ( sum(m.X[i,j]*Dij[(i,j)] for (i,j) in Dij)) #DISTANCIA
    return sum(Ck[k]*m.Y[i,k] for i in I for k in K) #COSTO
model.Obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)
#model.Obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# -------------------------------------------------
# Restricciones.

# -------------------------------------------------
# Σ Σ M_k * Y_i,k = N
def r1_rule(m):
    return sum(Mk[k]*m.Y[i,k] for i in I for k in K) == N
model.R1 = pyo.Constraint(rule=r1_rule)

# Σ_k Y_i,k ≤ 1  ∀i
def r2_rule(m, i):
    return sum(m.Y[i,k] for k in K) <= 1
model.R2 = pyo.Constraint(I, rule=r2_rule)


def r3_rule(m, i, j):
    if i < j:
        return m.X[i,j] <= sum(m.Y[i,k] for k in K)
    else:
        return pyo.Constraint.Skip
model.R3 = pyo.Constraint(I, J, rule=r3_rule)

def r4_rule(m, i, j):
    if i < j:
        return m.X[i,j] <= sum(m.Y[j,k] for k in K)
    else:
        return pyo.Constraint.Skip
model.R4 = pyo.Constraint(I, J, rule=r4_rule)

def r5_rule(m, i, j):
    if i < j:
        return m.X[i,j] >= sum(m.Y[i,k] + m.Y[j,k]for k in K)-1
    else:
        return pyo.Constraint.Skip
model.R5 = pyo.Constraint(I, J, rule=r5_rule)

def r6_rule(m):
  #return sum(Ck[k]*m.Y[i,k] for i in I for k in K)<= 19478.1
  return sum(m.X[i,j]*Dij[(i,j)] for (i,j) in Dij) >= 2*1578.39
model.R6 = pyo.Constraint(rule=r6_rule)

solver = pyo.SolverFactory("scip",executable=r"C:\Program Files\SCIPOptSuite 9.2.3\bin\scip.exe") 
solver.options["limits/gap"] = 0.01
results = solver.solve(model, tee=True)
# Mostrar resultados
model.Obj.display()

'pyomo.core.base.objective.ScalarObjective'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.objective.AbstractScalarObjective'>).
This is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
'pyomo.core.base.constraint.ScalarConstraint'>) on block unknown with a new
Component (type=<class
'pyomo.core.base.constraint.AbstractScalarConstraint'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.constraint.IndexedConstraint'>). This
is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.constraint.IndexedConstraint'>). This
is usually i

In [8]:
def evaluar_restriccion(constr, nombre):
    print(f"\n🔎 Evaluando {nombre}")
    print("-" * 60)

    for idx in constr:
        c = constr[idx]

        if not c.active:
            continue

        lhs = pyo.value(c.body)

        if c.has_lb():
            lb = pyo.value(c.lower)
            cumple_lb = lhs >= lb - 1e-6
            print(f"{nombre}{idx} | LHS={lhs:.4f} ≥ LB={lb:.4f} → {cumple_lb}")

        if c.has_ub():
            ub = pyo.value(c.upper)
            cumple_ub = lhs <= ub + 1e-6
            print(f"{nombre}{idx} | LHS={lhs:.4f} ≤ UB={ub:.4f} → {cumple_ub}")

In [79]:
evaluar_restriccion(model.R6, "R6")


🔎 Evaluando R6
------------------------------------------------------------
R6None | LHS=3160.7600 ≥ LB=3156.7800 → True


In [80]:
cestas_tipo_1 = sum(pyo.value(model.Y[i, 1]) for i in I)
cestas_tipo_2 = sum(pyo.value(model.Y[i, 2]) for i in I)

print(f"\nNúmero de cestas tipo 1 instaladas: {cestas_tipo_1}")
print(f"Número de cestas tipo 2 instaladas: {cestas_tipo_2}")


Número de cestas tipo 1 instaladas: 31.0
Número de cestas tipo 2 instaladas: 8.0


In [18]:
r6_lhs = sum(
    pyo.value(Ck[k] * model.Y[i, k])
    for i in I for k in K
)

print(f"\nCOSTO: {r6_lhs}")




COSTO: 43540.8


In [81]:
import os
print("\nGenerando archivo CSV con ubicaciones de las cestas instaladas...")

coords_df = pd.read_csv(r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\DISTANCIAS\17mutis.csv", sep=",")

# Separar columnas unidas
if 'i,j,Dij' in coords_df.columns:
    coords_df[['i','j','Dij']] = coords_df['i,j,Dij'].str.split(',', expand=True)
    coords_df['i'] = coords_df['i'].astype(int)
    coords_df['j'] = coords_df['j'].astype(int)

# --- Verificación de columnas disponibles ---
lat_i_col = 'lat_i'
lon_i_col = 'lon_i'
lat_j_col = 'lat_j' if 'lat_j' in coords_df.columns else None
lon_j_col = 'lon_j' if 'lon_j' in coords_df.columns else None

if lat_i_col not in coords_df.columns or lon_i_col not in coords_df.columns:
    raise ValueError("⚠️ El archivo original NO contiene columnas lat_i/lon_i necesarias.")

# --------------------------------------------------------
# Crear un repositorio unificado de coordenadas por nodo
# --------------------------------------------------------
coord_map = {}

# Registrar coords usando columna i
for _, row in coords_df.iterrows():
    i = int(row['i'])
    if i not in coord_map:
        coord_map[i] = {
            'lat': row[lat_i_col],
            'lon': row[lon_i_col]
        }

    # Registrar coords usando columna j si existen lat_j/lon_j
    if lat_j_col and lon_j_col:
        j = int(row['j'])
        if j not in coord_map:
            coord_map[j] = {
                'lat': row[lat_j_col],
                'lon': row[lon_j_col]
            }
    else:
        # Si no hay lat_j/lon_j, al menos asegura que j use los de i si no existe
        j = int(row['j'])
        if j not in coord_map:
            coord_map[j] = {
                'lat': row[lat_i_col],
                'lon': row[lon_i_col]
            }

# --------------------------------------------------------
# Crear lista de cestas instaladas
# --------------------------------------------------------
instaladas = []
for i in I:
    for k in K:
        if pyo.value(model.Y[i,k]) > 0.5:

            if i in coord_map:
                instaladas.append({
                    'nodo': int(i),
                    'tipo_cesta': int(k),
                    'latitud': coord_map[i]['lat'],
                    'longitud': coord_map[i]['lon']
                })
            else:
                print(f"⚠️ No se encontraron coordenadas para el nodo {i}. Revisar archivo original.")

# Crear DataFrame
df_instaladas = pd.DataFrame(instaladas)

# Ordenar por nodo
if not df_instaladas.empty:
    df_instaladas = df_instaladas.sort_values(by=['nodo']).reset_index(drop=True)

# Guardar resultado
carpeta_salida = r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\SALIDAS\17MUTIS"
os.makedirs(carpeta_salida, exist_ok=True)

ruta_salida = os.path.join(carpeta_salida, "sol_mutis_E3.csv")
df_instaladas.to_csv(ruta_salida, index=False)

print("\n✔ Archivo generado exitosamente:")
print(ruta_salida)

df_instaladas.head()


Generando archivo CSV con ubicaciones de las cestas instaladas...

✔ Archivo generado exitosamente:
C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\SALIDAS\17MUTIS\sol_mutis_E3.csv


,nodo,tipo_cesta,latitud,longitud
0,16,1,7.084697,-73.144226
1,65,1,7.090526,-73.137538
2,80,2,7.084984,-73.143423
3,82,1,7.084475,-73.145103
4,83,1,7.095334,-73.126888


In [12]:
pip install geopandas requests osmnx shapely

Note: you may need to restart the kernel to use updated packages.


In [13]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
from shapely.geometry import Point
from shapely.ops import unary_union
from geopy.distance import geodesic
import numpy as np

In [14]:
import sys
print(sys.executable)

c:\Users\UIS\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [82]:
import pandas as pd
import folium

df= pd.read_csv(r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\COORDENADAS\17mutis.csv", sep=";")

gdf_existentes = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.Longitud, df.Latitud), # Correct order for x, y
    crs="EPSG:4326"
)

df_nuevas= pd.read_csv(r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\SALIDAS\17MUTIS\sol_mutis_E3.csv", sep=",")

if 'gdf_existentes' in globals() and len(gdf_existentes) > 0:
    center_lat = gdf_existentes.geometry.y.iloc[0]
    center_lon = gdf_existentes.geometry.x.iloc[0]
else:
    center_lat = df_nuevas.loc[0, "latitud"]
    center_lon = df_nuevas.loc[0, "longitud"]

mapa = folium.Map(location=[center_lat, center_lon], zoom_start=15)

feature_existentes = folium.FeatureGroup(name="Cestas Existentes")

for _, row in gdf_existentes.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2.5,
        color='blue',
        fill=True,
        fill_opacity=0.8,
        popup='Existente'
    ).add_to(feature_existentes)

feature_existentes.add_to(mapa)

feature_nuevas = folium.FeatureGroup(name="Cestas Nuevas")

for _, row in df_nuevas.iterrows():
    # Assign color based on 'tipo_cesta'
    color = 'red' if row["tipo_cesta"] == 1 else 'purple'
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=2.5,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=f"Nueva cesta Tipo {row['tipo_cesta']}"
    ).add_to(feature_nuevas)

feature_nuevas.add_to(mapa)

folium.LayerControl().add_to(mapa)


mapa.save("mapa.html")
import webbrowser
webbrowser.open("mapa.html")

True

In [1]:
%pip install osmnx geopandas folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# ============================================
# 1) IMPORTS
# ============================================
import pandas as pd
import geopandas as gpd
import folium
from pathlib import Path
import webbrowser
from shapely.geometry import Polygon, MultiPolygon

# OSMnx y compatibilidad de versión
try:
    import osmnx as ox
except ImportError as e:
    raise ImportError(
        "No se encontró el paquete 'osmnx'. Instálalo con:\n\n"
        "%pip install osmnx geopandas folium packaging\n\n"
        "y vuelve a ejecutar esta celda."
    ) from e

from packaging import version

# ============================================
# 2) PARÁMETROS (AJUSTA ESTAS RUTAS/OPCIONES)
# ============================================
CARPETA_EXISTENTES = Path(r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\COORDENADAS")
CARPETA_NUEVAS     = Path(r"C:\Users\UIS\OneDrive - Universidad Industrial de Santander\PROYECTO DE INVESTIGACIÓN PERIODO DE PRUEBA\modelo\SALIDAS\E3")

SEP_EXISTENTES = ";"
SEP_NUEVAS = ","

CACHE_COMUNAS_GEOJSON = Path("bucaramanga_comunas.geojson")
CACHE_UNION_GEOJSON   = Path("bucaramanga_union_comunas.geojson")

FILTRAR_DENTRO = True
LIMITAR_NAVEGACION = True

# ============================================
# 3) HELPERS
# ============================================
def leer_csvs(carpeta: Path, sep: str, encoding_preferido: str = "utf-8-sig") -> pd.DataFrame:
    if not carpeta.exists():
        print(f"[AVISO] La carpeta no existe: {carpeta}")
        return pd.DataFrame()
    archivos = sorted(carpeta.glob("*.csv"))
    if not archivos:
        print(f"[AVISO] No se encontraron CSV en: {carpeta}")
        return pd.DataFrame()
    dfs = []
    for f in archivos:
        try:
            df = pd.read_csv(f, sep=sep, encoding=encoding_preferido)
        except UnicodeDecodeError:
            df = pd.read_csv(f, sep=sep, encoding="latin-1")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def _fix_invalid_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.empty:
        return gdf
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.buffer(0)
    return gdf

def osm_features_from_polygon(polygon, tags: dict):
    """
    Compatibilidad OSMnx 1.x y 2.x.
    - 1.x -> ox.geometries_from_polygon
    - 2.x -> ox.features_from_polygon
    """
    ox_version = version.parse(ox.__version__)
    if ox_version.major >= 2:
        return ox.features_from_polygon(polygon, tags=tags)
    else:
        return ox.geometries_from_polygon(polygon, tags=tags)

def obtener_comunas_y_union(place: str = "Bucaramanga, Santander, Colombia",
                            cache_comunas: Path = CACHE_COMUNAS_GEOJSON,
                            cache_union: Path = CACHE_UNION_GEOJSON) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    # Intentar caché
    gdf_comunas = None
    gdf_union   = None
    if cache_comunas.exists():
        try:
            gdf_comunas = gpd.read_file(cache_comunas)
            if gdf_comunas.crs is None or gdf_comunas.crs.to_epsg() != 4326:
                gdf_comunas = gdf_comunas.to_crs(epsg=4326)
        except Exception as e:
            print(f"[AVISO] No se pudo leer cache de comunas ({cache_comunas}): {e}")
    if cache_union.exists():
        try:
            gdf_union = gpd.read_file(cache_union)
            if gdf_union.crs is None or gdf_union.crs.to_epsg() != 4326:
                gdf_union = gdf_union.to_crs(epsg=4326)
        except Exception as e:
            print(f"[AVISO] No se pudo leer cache de unión ({cache_union}): {e}")

    if gdf_comunas is not None and gdf_union is not None and not gdf_comunas.empty and not gdf_union.empty:
        return gdf_comunas, gdf_union

    # Polígono contenedor de Bucaramanga
    gdf_buca = ox.geocode_to_gdf(place).to_crs(epsg=4326)
    buca_poly = gdf_buca.geometry.iloc[0]

    # Traer límites administrativos dentro del polígono (comunas/barrios)
    tags = {"boundary": "administrative", "admin_level": ["9", "10"]}
    gdf = osm_features_from_polygon(buca_poly, tags)
    if gdf.empty:
        raise RuntimeError("No se obtuvieron límites administrativos (comunas/barrios) desde OSM. Intenta más tarde.")

    gdf = gdf.to_crs(epsg=4326)
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]

    # Identificar comunas por nombre
    def es_comuna(row) -> bool:
        for col in ["name", "Name", "NAME"]:
            if col in row and isinstance(row[col], str) and "COMUNA" in row[col].upper():
                return True
        return False

    gdf["__es_comuna__"] = gdf.apply(es_comuna, axis=1)
    gdf_comunas = gdf[gdf["__es_comuna__"]].copy()
    if gdf_comunas.empty:
        # Fallback: admin_level=9
        gdf_comunas = gdf[gdf["admin_level"].astype(str) == "9"].copy()
        print("[AVISO] No se halló 'Comuna' en el nombre. Usando admin_level=9 como aproximación.")

    # Intersecar con BGA para recortar geometrías “fugitivas”
    gdf_comunas = gpd.overlay(gdf_comunas, gdf_buca[["geometry"]], how="intersection")
    gdf_comunas = _fix_invalid_geometries(gdf_comunas)

    # Mantener columnas útiles
    name_col = "name" if "name" in gdf_comunas.columns else None
    keep_cols = ["geometry"]
    if name_col:
        keep_cols.insert(0, name_col)
    gdf_comunas = gdf_comunas[keep_cols].copy()

    # Orden por número si aplica (Comuna 1, Comuna 2, …)
    if name_col:
        import re
        def extrae_numero(n):
            if isinstance(n, str):
                m = re.search(r"(\d+)", n)
                return int(m.group(1)) if m else 9999
            return 9999
        gdf_comunas["__orden__"] = gdf_comunas[name_col].apply(extrae_numero)
        gdf_comunas = gdf_comunas.sort_values("__orden__").drop(columns="__orden__", errors="ignore")

    # Aviso si no son 17
    n = len(gdf_comunas)
    if n != 17:
        print(f"[AVISO] Se encontraron {n} comunas en OSM (esperado: 17). Se usará la unión de las encontradas.")

    # Unión
    union_geom = gdf_comunas.unary_union
    if isinstance(union_geom, (Polygon, MultiPolygon)):
        gdf_union = gpd.GeoDataFrame({"name": ["Unión Comunas BGA"], "geometry": [union_geom]}, crs="EPSG:4326")
    else:
        raise RuntimeError("No se pudo construir la unión de comunas.")

    # Guardar cachés
    try:
        gdf_comunas.to_file(cache_comunas, driver="GeoJSON")
        print(f"[OK] Comunas guardadas en: {cache_comunas}")
    except Exception as e:
        print(f"[AVISO] No se pudo guardar cache de comunas: {e}")
    try:
        gdf_union.to_file(cache_union, driver="GeoJSON")
        print(f"[OK] Unión de comunas guardada en: {cache_union}")
    except Exception as e:
        print(f"[AVISO] No se pudo guardar cache de unión: {e}")

    return gdf_comunas, gdf_union

# ============================================
# 4) CARGA MASIVA DE TUS CAPAS (EXISTENTES/NUEVAS)
# ============================================
df_existentes = leer_csvs(CARPETA_EXISTENTES, sep=SEP_EXISTENTES)
df_nuevas     = leer_csvs(CARPETA_NUEVAS,     sep=SEP_NUEVAS)

# Validaciones
cols_req_existentes = {"Latitud", "Longitud"}
if not df_existentes.empty and not cols_req_existentes.issubset(df_existentes.columns):
    raise ValueError(f"Las EXISTENTES deben tener columnas {cols_req_existentes}. Encontradas: {list(df_existentes.columns)}")

cols_req_nuevas = {"latitud", "longitud", "tipo_cesta"}
if not df_nuevas.empty and not cols_req_nuevas.issubset(df_nuevas.columns):
    raise ValueError(f"Las NUEVAS deben tener columnas {cols_req_nuevas}. Encontradas: {list(df_nuevas.columns)}")

# GeoDataFrames
if not df_existentes.empty:
    gdf_existentes = gpd.GeoDataFrame(
        df_existentes,
        geometry=gpd.points_from_xy(df_existentes["Longitud"], df_existentes["Latitud"]),
        crs="EPSG:4326"
    )
else:
    gdf_existentes = gpd.GeoDataFrame(df_existentes.copy(), geometry=[], crs="EPSG:4326")

if not df_nuevas.empty:
    gdf_nuevas = gpd.GeoDataFrame(
        df_nuevas.copy(),
        geometry=gpd.points_from_xy(df_nuevas["longitud"], df_nuevas["latitud"]),
        crs="EPSG:4326"
    )
else:
    gdf_nuevas = gpd.GeoDataFrame(df_nuevas.copy(), geometry=[], crs="EPSG:4326")

# ============================================
# 5) OBTENER COMUNAS Y UNIÓN
# ============================================
gdf_comunas, gdf_union = obtener_comunas_y_union()

minx, miny, maxx, maxy = gdf_union.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# ============================================
# 6) MAPA BASE + CAPAS POLIGONALES
# ============================================
mapa = folium.Map(location=[center_lat, center_lon], zoom_start=15, prefer_canvas=True, tiles=None)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png",
    attr='&copy; OpenStreetMap contributors &copy; CARTO',
    name="Base sin etiquetas",
    control=False
).add_to(mapa)

def estilo_comunas(_):
    return {"color": "#555555", "weight": 1.5, "fillColor": "#DDDDDD", "fillOpacity": 0.05}

folium.GeoJson(
    gdf_comunas,
    name="Comunas Bucaramanga",
    style_function=lambda _: {"color": "#AAAAAA", "weight": 0.8, "fillOpacity": 0}
    # highlight_function eliminado
    # tooltip eliminado
).add_to(mapa)


folium.GeoJson(
    gdf_union,
    name="Unión Comunas (Municipio)",
    style_function=lambda f: {
        "color": "#2E86C1",
        "weight": 0.5,
        "fillColor": "#85C1E9",
        "fillOpacity": 0.15
    }
).add_to(mapa)

mapa.fit_bounds([[miny, minx], [maxy, maxx]])

if LIMITAR_NAVEGACION:
    margen = 0.02
    mapa.options.update(maxBounds=[[miny - margen, minx - margen], [maxy + margen, maxx + margen]])

# ============================================
# 7) (OPCIONAL) FILTRAR PUNTOS DENTRO DE LA UNIÓN
# ============================================
if FILTRAR_DENTRO:
    if not gdf_existentes.empty:
        gdf_existentes = gpd.sjoin(
            gdf_existentes,
            gdf_union[["geometry"]],
            predicate="within",
            how="inner"
        ).drop(columns=["index_right"], errors="ignore")
    if not gdf_nuevas.empty:
        gdf_nuevas = gpd.sjoin(
            gdf_nuevas,
            gdf_union[["geometry"]],
            predicate="within",
            how="inner"
        ).drop(columns=["index_right"], errors="ignore")

# ============================================
# 8) CAPAS DE PUNTOS (TUS COLORES)
# ============================================
feature_existentes = folium.FeatureGroup(name="Cestas Existentes")
if not gdf_existentes.empty:
    for _, row in gdf_existentes.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=1.5,
            color='blue',
            fill=True,
            fill_opacity=0.8,
            weight=0,            # <-- sin borde
            stroke=False,        # <-- sin contorno
            popup='Existente'
        ).add_to(feature_existentes)
feature_existentes.add_to(mapa)

feature_nuevas = folium.FeatureGroup(name="Cestas Nuevas")
if not gdf_nuevas.empty:
    for _, row in gdf_nuevas.iterrows():
        tipo = row.get("tipo_cesta", None)
        try:
            color = 'red' if int(tipo) == 1 else 'purple'
        except Exception:
            color = 'purple'
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=1.5,
            color=color,
            fill=True,
            fill_opacity=0.8,   # <-- menos saturación
            opacity=0.8,        # <-- bordes más tenues
            weight=0,            # <-- sin borde
            stroke=False,        # <-- sin contorno
            popup=f"Nueva cesta Tipo {tipo}" if tipo is not None else "Nueva cesta"
        ).add_to(feature_nuevas)
feature_nuevas.add_to(mapa)

from branca.element import Element
legend = Element("""
<div style="
    position: fixed;
    bottom: 0px;
    right: 0px;
    z-index: 9999;
    background-color: rgba(255, 255, 255, 0.92);
    padding: 10px 12px;
    border: 1px solid #ccc;
    border-radius: 6px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15);
    font-size: 12px;">
  <b> </b><br>
  <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#ff0000;margin-right:6px;"></span> Cestas tipo 1 <br>
  <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#800080;margin-right:6px;"></span> Cestas tipo 2 <br>
  <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#0000ff;margin-right:6px;"></span> Cestas existentes
</div>
""")
mapa.get_root().html.add_child(legend)

folium.LayerControl().add_to(mapa)

# ============================================
# 9) SALIDA
# ============================================
salida_html = "mapa.html"
mapa.save(salida_html)
webbrowser.open(salida_html)
print(f"Mapa generado: {salida_html}\nOSMnx versión usada: {ox.__version__}")

Mapa generado: mapa.html
OSMnx versión usada: 2.0.7
